# Trial Demo
Run a trial end-to-end: create a trial, create a fixed route, record frames + risk, optionally append model risk, and generate a video with a risk curve tile.

In [1]:
# Imports
import os
import time
from pathlib import Path

from MIREIA.config import Config
from MIREIA.simulation.world_manager import WorldManager

In [2]:
# Launch CARLA (skip if already running)
import subprocess
subprocess.Popen("CarlaUE4.exe", shell=True)

<Popen: returncode: None args: 'CarlaUE4.exe'>

In [4]:
import random
import numpy as np
import carla
from MIREIA.config import Config
from MIREIA.simulation.traffic_handler import TrafficHandler
from MIREIA.simulation.simple_route_controller import SimpleRouteController
from MIREIA.simulation.bridge import SimulationBridge
from MIREIA.simulation.world_manager import WorldManager

# reload packages (for development)
import importlib

# Direct run: spawn ego + attach controller (no Scenario, no WorldManager)
client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)
world = client.get_world()
world_map = world.get_map()

# Keep previous world settings to restore later
prev_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

spawn_points = world_map.get_spawn_points()
if len(spawn_points) < 2:
    raise RuntimeError("Need at least 2 spawn points to run a route.")

# Randomize ego spawn position each execution
ego_spawn_index = random.randrange(len(spawn_points))

handler = TrafficHandler(client, world, seed=42)
base_speed_kmh = 20.0
controller = SimpleRouteController(target_speed=base_speed_kmh, sampling_resolution=2.0)

# Demo: modify target speed every tick from the base speed.
# Signature: fn(base_speed_kmh, tick_index, controller) -> target_speed_kmh
def demo_speed_modifier(base_speed, tick_idx, _controller):
    phase = (tick_idx // 120) % 3
    if phase == 0:
        return 0.5 * base_speed   # slower block
    if phase == 1:
        return 1.0 * base_speed   # nominal block
    return 2.0 * base_speed       # faster block

controller.set_speed_modifier(demo_speed_modifier)

ego = handler.spawn_ego(
    blueprint_id="vehicle.lincoln.mkz_2020",
    spawn_index=ego_spawn_index,
    autopilot=False,
    controller=controller,
 )

if hasattr(carla, "VehicleLightState"):
    light_state = (
        carla.VehicleLightState.Position
        | carla.VehicleLightState.LowBeam
        | carla.VehicleLightState.Interior
    )
    ego.set_light_state(carla.VehicleLightState(light_state))

# Let CARLA settle one frame so ego transform and waypoint are consistent
world.tick()

start = ego.get_location()

# Build valid destination candidates, then choose one randomly each execution
candidates = []
for i, sp in enumerate(spawn_points):
    if i == ego_spawn_index:
        continue
    d = start.distance(sp.location)
    if d > 35.0:
        candidates.append((d, i))

if not candidates:
    raise RuntimeError("No suitable destination spawn point found.")

_, best_idx = random.choice(candidates)
end = spawn_points[best_idx].location
controller.set_destination(start, end)
print(f"Plan length: {controller.get_plan_length()} waypoints")

print(f"Spawn index: {ego_spawn_index}")
print(f"Start: ({start.x:.1f}, {start.y:.1f})")
print(f"End:   ({end.x:.1f}, {end.y:.1f})  [spawn index {best_idx}]")
print("Speed profile phases: 0.5x -> 1.0x -> 2.0x (changes every 120 ticks)")

# Risk setup with WorldManager: bake static first, then query per-tick risk
bridge = SimulationBridge(world)
wm_risk = WorldManager(sync_mode=True, fixed_delta=0.05, verbose=False)
wm_risk.world = world
wm_risk.bridge = bridge
wm_risk._WorldManager__bake_static_risk_map()
risk_values = []

def draw_ego_glow(vehicle: carla.Vehicle):
    t = vehicle.get_transform()
    loc = t.location
    extent = vehicle.bounding_box.extent
    world.debug.draw_box(
        box=carla.BoundingBox(loc, extent),
        rotation=t.rotation,
        thickness=0.15,
        color=carla.Color(0, 255, 255),
        life_time=0.08,
    )
    world.debug.draw_point(
        location=carla.Location(x=loc.x, y=loc.y, z=loc.z + 1.0),
        size=0.2,
        color=carla.Color(255, 255, 0),
        life_time=0.08,
    )

def draw_route_endpoints(start_loc: carla.Location, end_loc: carla.Location):
    world.debug.draw_point(
        location=carla.Location(x=start_loc.x, y=start_loc.y, z=start_loc.z + 0.5),
        size=0.25,
        color=carla.Color(255, 0, 0),
        life_time=0.08,
    )
    world.debug.draw_point(
        location=carla.Location(x=end_loc.x, y=end_loc.y, z=end_loc.z + 0.5),
        size=0.25,
        color=carla.Color(255, 0, 0),
        life_time=0.08,
    )

for step in range(10000):
    handler.run_ego_controller_step()
    world.tick()
    bridge.update()
    try:
        risk_now = wm_risk.get_risk()
        risk_values.append(float(risk_now))
    except RuntimeError:
        pass

    draw_ego_glow(ego)
    draw_route_endpoints(start, end)
    controller.draw_plan(world, max_points=1400, life_time=0.08)
    if step % 60 == 0:
        print(f"step={step:04d} target_speed={controller.get_last_applied_target_speed():.2f} km/h")
    if controller.done():
        print(f"Route finished at step {step}")
        for i in range(100):
            draw_ego_glow(ego)
            draw_route_endpoints(start, end)
            world.tick()
        break

dt = settings.fixed_delta_seconds or 0.05
if risk_values:
    total_risk_auc = float(np.trapz(np.array(risk_values, dtype=np.float64), dx=dt))
    print(f"Total route risk AUC: {total_risk_auc:.4f}")
    print(f"Risk samples: {len(risk_values)}  duration: {len(risk_values) * dt:.2f}s")
else:
    print("No risk samples collected.")

handler.destroy_all()
world.apply_settings(prev_settings)
print("Done.")

Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index 15 (autopilot=False)
Plan length: 219 waypoints
Spawn index: 15
Start: (-56.9, 140.5)
End:   (-5.8, -64.5)  [spawn index 81]
Speed profile phases: 0.5x -> 1.0x -> 2.0x (changes every 120 ticks)
step=0000 target_speed=10.00 km/h
step=0060 target_speed=10.00 km/h
step=0120 target_speed=20.00 km/h
step=0180 target_speed=20.00 km/h
step=0240 target_speed=40.00 km/h
step=0300 target_speed=40.00 km/h
step=0360 target_speed=10.00 km/h
step=0420 target_speed=10.00 km/h
step=0480 target_speed=20.00 km/h
step=0540 target_speed=20.00 km/h
step=0600 target_speed=40.00 km/h
step=0660 target_speed=40.00 km/h
step=0720 target_speed=10.00 km/h
step=0780 target_speed=10.00 km/h
step=0840 target_speed=20.00 km/h
step=0900 target_speed=20.00 km/h
step=0960 target_speed=40.00 km/h
step=1020 target_speed=40.00 km/h
step=1080 target_speed=10.00 km/h
step=1140 target_speed=10.00 km/h
step=1200 target_speed=20.00 km/h
step=1260 target_speed=20.00 km/h
st

In [5]:
# Fixed-route comparison: same manual route (by carla.Location), different speed multipliers (0.5x vs 2.0x)
import numpy as np
import carla
from MIREIA.config import Config
from MIREIA.simulation.traffic_handler import TrafficHandler
from MIREIA.simulation.simple_route_controller import SimpleRouteController
from MIREIA.simulation.bridge import SimulationBridge
from MIREIA.simulation.world_manager import WorldManager

client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)
world = client.get_world()
world_map = world.get_map()
spawn_points = world_map.get_spawn_points()
if len(spawn_points) < 2:
    raise RuntimeError("Need at least 2 spawn points for fixed-route comparison.")

prev_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

# Manual route definition by coordinates.
# You can use carla.Location(...) or carla.Transform(carla.Location(...), carla.Rotation()).
START_CARLA = carla.Location(x=7.409, y=-64.400, z=0.000)
END_CARLA = carla.Location(x=44.003, y=50.549, z=0.000)

def as_location(value):
    if isinstance(value, carla.Transform):
        return value.location
    if isinstance(value, carla.Location):
        return value
    raise TypeError("START_CARLA and END_CARLA must be carla.Location or carla.Transform")

raw_start_loc = as_location(START_CARLA)
raw_end_loc = as_location(END_CARLA)

# Project points to driving lane to get route-ready points and safe spawn orientation.
start_wp = world_map.get_waypoint(
    carla.Location(x=raw_start_loc.x, y=raw_start_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
 )
end_wp = world_map.get_waypoint(
    carla.Location(x=raw_end_loc.x, y=raw_end_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
 )
if start_wp is None or end_wp is None:
    raise RuntimeError("Could not project START_CARLA/END_CARLA to a driving lane")

start_tf = start_wp.transform
start_tf.location.z += 0.5
start_loc = start_tf.location
end_loc = end_wp.transform.location

print(f"Manual route input start: ({raw_start_loc.x:.3f}, {raw_start_loc.y:.3f}, {raw_start_loc.z:.3f})")
print(f"Manual route input end:   ({raw_end_loc.x:.3f}, {raw_end_loc.y:.3f}, {raw_end_loc.z:.3f})")
print(f"Projected start: ({start_loc.x:.3f}, {start_loc.y:.3f}, {start_loc.z:.3f})")
print(f"Projected end:   ({end_loc.x:.3f}, {end_loc.y:.3f}, {end_loc.z:.3f})")

def nearest_spawn_indices(target_loc: carla.Location) -> list[int]:
    dists = [(target_loc.distance(sp.location), i) for i, sp in enumerate(spawn_points)]
    dists.sort(key=lambda x: x[0])
    return [i for _, i in dists]

def run_fixed_route(multiplier: float, base_speed_kmh: float = 20.0, max_steps: int = 10000) -> dict:
    handler = TrafficHandler(client, world, seed=42)
    controller = SimpleRouteController(target_speed=base_speed_kmh, sampling_resolution=2.0)

    def speed_modifier(base_speed, tick_idx, _controller):
        return multiplier * base_speed

    controller.set_speed_modifier(speed_modifier)

    ego = None
    try:
        ego = handler.spawn_ego(
            blueprint_id="vehicle.lincoln.mkz_2020",
            spawn_point=start_tf,
            autopilot=False,
            controller=controller,
        )
    except RuntimeError as exc:
        print(f"Exact transform spawn failed: {exc}")

    if ego is None:
        for fallback_idx in nearest_spawn_indices(start_loc)[:20]:
            try:
                ego = handler.spawn_ego(
                    blueprint_id="vehicle.lincoln.mkz_2020",
                    spawn_index=fallback_idx,
                    autopilot=False,
                    controller=controller,
                )
                print(f"Spawn fallback succeeded at spawn_index={fallback_idx}")
                break
            except RuntimeError:
                continue

    if ego is None:
        raise RuntimeError("Failed to spawn ego vehicle: start position is blocked. Try another START_CARLA.")

    if hasattr(carla, "VehicleLightState"):
        light_state = (
            carla.VehicleLightState.Position
            | carla.VehicleLightState.LowBeam
            | carla.VehicleLightState.Interior
        )
        ego.set_light_state(carla.VehicleLightState(light_state))

    world.tick()
    start = ego.get_location()
    controller.set_destination(start, end_loc)

    bridge = SimulationBridge(world)
    wm_risk = WorldManager(sync_mode=True, fixed_delta=0.05, verbose=False)
    wm_risk.world = world
    wm_risk.bridge = bridge
    wm_risk._WorldManager__bake_static_risk_map()

    risks = []
    positions = []
    finished_step = None

    for step in range(max_steps):
        handler.run_ego_controller_step()
        world.tick()
        bridge.update()

        ego_kin = bridge.get_ego_kinematics()
        if ego_kin is not None:
            positions.append((float(ego_kin.x), float(ego_kin.y)))

        try:
            risks.append(float(wm_risk.get_risk()))
        except RuntimeError:
            pass

        # Visualize current planned path (same style as Cell 4).
        controller.draw_plan(world, max_points=1400, life_time=0.08)

        if controller.done():
            finished_step = step
            break

    # Traveled distance (meters) from sampled ego trajectory.
    if len(positions) >= 2:
        pos_arr = np.array(positions, dtype=np.float64)
        seg = np.linalg.norm(np.diff(pos_arr, axis=0), axis=1)
        traveled_m = float(seg.sum())
    else:
        seg = np.array([], dtype=np.float64)
        traveled_m = 0.0

    # Distance-integrated risk: sum(average segment risk * segment length).
    risk_arr = np.array(risks, dtype=np.float64) if risks else np.array([], dtype=np.float64)
    if seg.size > 0 and risk_arr.size >= 2:
        n = min(seg.size, risk_arr.size - 1)
        risk_integral_distance = float(np.sum(0.5 * (risk_arr[:n] + risk_arr[1:n + 1]) * seg[:n]))
    else:
        risk_integral_distance = float("nan")

    risk_per_meter = (risk_integral_distance / traveled_m) if traveled_m > 0 and np.isfinite(risk_integral_distance) else float("nan")
    dt = settings.fixed_delta_seconds or 0.05

    handler.destroy_all()

    return {
        "multiplier": multiplier,
        "target_speed_kmh": multiplier * base_speed_kmh,
        "steps": finished_step if finished_step is not None else max_steps,
        "num_samples": len(risks),
        "duration_s": len(risks) * dt,
        "traveled_m": traveled_m,
        "risk_integral_distance": risk_integral_distance,
        "risk_per_meter": risk_per_meter,
    }

results = []
for m in (0.5, 2.0):
    res = run_fixed_route(m, base_speed_kmh=20.0)
    results.append(res)
    print(
        f"mult={res['multiplier']:.1f}x  target={res['target_speed_kmh']:.1f} km/h  "
        f"risk/m={res['risk_per_meter']:.6f}  distance={res['traveled_m']:.2f}m  "
        f"samples={res['num_samples']}  duration={res['duration_s']:.2f}s"
    )

if len(results) == 2:
    a = results[0]["risk_per_meter"]
    b = results[1]["risk_per_meter"]
    if np.isfinite(a) and np.isfinite(b) and a != 0:
        print(f"risk/m ratio (2.0x / 0.5x): {b / a:.4f}")

world.apply_settings(prev_settings)
print("Fixed-route comparison done.")

Manual route input start: (7.409, -64.400, 0.000)
Manual route input end:   (44.003, 50.549, 0.000)
Projected start: (7.409, -64.400, 0.500)
Projected end:   (44.003, 50.549, 0.000)
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False)
Destroyed ego vehicle.
mult=0.5x  target=10.0 km/h  risk/m=1.571654  distance=364.60m  samples=3497  duration=174.85s
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False)
Destroyed ego vehicle.
mult=2.0x  target=40.0 km/h  risk/m=2.124164  distance=362.38m  samples=1054  duration=52.70s
risk/m ratio (2.0x / 0.5x): 1.3515
Fixed-route comparison done.


In [ ]:
# Same fixed-route comparison, but with extra traffic before route execution
import numpy as np
import carla
from MIREIA.config import Config
from MIREIA.simulation.traffic_handler import TrafficHandler
from MIREIA.simulation.simple_route_controller import SimpleRouteController
from MIREIA.simulation.bridge import SimulationBridge
from MIREIA.simulation.world_manager import WorldManager

if "START_CARLA" not in globals() or "END_CARLA" not in globals():
    raise RuntimeError("Run Cell 5 first so START_CARLA and END_CARLA are defined.")

def as_location(value):
    if isinstance(value, carla.Transform):
        return value.location
    if isinstance(value, carla.Location):
        return value
    raise TypeError("START_CARLA and END_CARLA must be carla.Location or carla.Transform")

client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)
world = client.get_world()
world_map = world.get_map()
spawn_points = world_map.get_spawn_points()
if len(spawn_points) < 2:
    raise RuntimeError("Need at least 2 spawn points for fixed-route comparison.")

prev_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

raw_start_loc = as_location(START_CARLA)
raw_end_loc = as_location(END_CARLA)

start_wp = world_map.get_waypoint(
    carla.Location(x=raw_start_loc.x, y=raw_start_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
 )
end_wp = world_map.get_waypoint(
    carla.Location(x=raw_end_loc.x, y=raw_end_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
 )
if start_wp is None or end_wp is None:
    raise RuntimeError("Could not project START_CARLA/END_CARLA to a driving lane")

start_tf = start_wp.transform
start_tf.location.z += 0.5
start_loc = start_tf.location
end_loc = end_wp.transform.location

print(f"Using START_CARLA: ({start_loc.x:.3f}, {start_loc.y:.3f}, {start_loc.z:.3f})")
print(f"Using END_CARLA:   ({end_loc.x:.3f}, {end_loc.y:.3f}, {end_loc.z:.3f})")

N_TRAFFIC_VEHICLES = 45
BASE_SPEED_KMH = 20.0

def nearest_spawn_indices(target_loc: carla.Location) -> list[int]:
    dists = [(target_loc.distance(sp.location), i) for i, sp in enumerate(spawn_points)]
    dists.sort(key=lambda x: x[0])
    return [i for _, i in dists]

def run_fixed_route_with_traffic(multiplier: float, max_steps: int = 10000) -> dict:
    handler = TrafficHandler(client, world, seed=42)
    controller = SimpleRouteController(target_speed=BASE_SPEED_KMH, sampling_resolution=2.0)

    def speed_modifier(base_speed, tick_idx, _controller):
        return multiplier * base_speed

    controller.set_speed_modifier(speed_modifier)

    ego = None
    try:
        ego = handler.spawn_ego(
            blueprint_id="vehicle.lincoln.mkz_2020",
            spawn_point=start_tf,
            autopilot=False,
            controller=controller,
        )
    except RuntimeError as exc:
        print(f"Exact transform spawn failed: {exc}")

    if ego is None:
        for fallback_idx in nearest_spawn_indices(start_loc)[:20]:
            try:
                ego = handler.spawn_ego(
                    blueprint_id="vehicle.lincoln.mkz_2020",
                    spawn_index=fallback_idx,
                    autopilot=False,
                    controller=controller,
                )
                print(f"Spawn fallback succeeded at spawn_index={fallback_idx}")
                break
            except RuntimeError:
                continue

    if ego is None:
        raise RuntimeError("Failed to spawn ego vehicle: start position is blocked. Try another START_CARLA.")

    # Add extra background traffic before route execution.
    spawned_traffic = handler.spawn_vehicles(n=N_TRAFFIC_VEHICLES, safe=True, car_lights_on=False)
    print(f"Spawned traffic vehicles: {len(spawned_traffic)}")

    if hasattr(carla, "VehicleLightState"):
        light_state = (
            carla.VehicleLightState.Position
            | carla.VehicleLightState.LowBeam
            | carla.VehicleLightState.Interior
        )
        ego.set_light_state(carla.VehicleLightState(light_state))

    world.tick()
    start = ego.get_location()
    controller.set_destination(start, end_loc)

    bridge = SimulationBridge(world)
    wm_risk = WorldManager(sync_mode=True, fixed_delta=0.05, verbose=False)
    wm_risk.world = world
    wm_risk.bridge = bridge
    wm_risk._WorldManager__bake_static_risk_map()

    risks = []
    positions = []
    finished_step = None

    for step in range(max_steps):
        handler.run_ego_controller_step()
        world.tick()
        bridge.update()

        ego_kin = bridge.get_ego_kinematics()
        if ego_kin is not None:
            positions.append((float(ego_kin.x), float(ego_kin.y)))

        try:
            risks.append(float(wm_risk.get_risk()))
        except RuntimeError:
            pass

        # Visualize current planned path (same style as Cell 4).
        controller.draw_plan(world, max_points=1400, life_time=0.08)

        if controller.done():
            finished_step = step
            break

    if len(positions) >= 2:
        pos_arr = np.array(positions, dtype=np.float64)
        seg = np.linalg.norm(np.diff(pos_arr, axis=0), axis=1)
        traveled_m = float(seg.sum())
    else:
        seg = np.array([], dtype=np.float64)
        traveled_m = 0.0

    risk_arr = np.array(risks, dtype=np.float64) if risks else np.array([], dtype=np.float64)
    if seg.size > 0 and risk_arr.size >= 2:
        n = min(seg.size, risk_arr.size - 1)
        risk_integral_distance = float(np.sum(0.5 * (risk_arr[:n] + risk_arr[1:n + 1]) * seg[:n]))
    else:
        risk_integral_distance = float("nan")

    risk_per_meter = (risk_integral_distance / traveled_m) if traveled_m > 0 and np.isfinite(risk_integral_distance) else float("nan")
    dt = settings.fixed_delta_seconds or 0.05

    handler.destroy_all()

    return {
        "multiplier": multiplier,
        "target_speed_kmh": multiplier * BASE_SPEED_KMH,
        "steps": finished_step if finished_step is not None else max_steps,
        "num_samples": len(risks),
        "duration_s": len(risks) * dt,
        "traveled_m": traveled_m,
        "risk_integral_distance": risk_integral_distance,
        "risk_per_meter": risk_per_meter,
    }

results_traffic = []
for m in (0.5, 2.0):
    res = run_fixed_route_with_traffic(m)
    results_traffic.append(res)
    print(
        f"[traffic] mult={res['multiplier']:.1f}x  target={res['target_speed_kmh']:.1f} km/h  "
        f"risk/m={res['risk_per_meter']:.6f}  distance={res['traveled_m']:.2f}m  "
        f"samples={res['num_samples']}  duration={res['duration_s']:.2f}s"
    )

if len(results_traffic) == 2:
    a = results_traffic[0]["risk_per_meter"]
    b = results_traffic[1]["risk_per_meter"]
    if np.isfinite(a) and np.isfinite(b) and a != 0:
        print(f"[traffic] risk/m ratio (2.0x / 0.5x): {b / a:.4f}")

world.apply_settings(prev_settings)
print("Fixed-route comparison with extra traffic done.")

Using START_CARLA: (7.409, -64.400, 0.500)
Using END_CARLA:   (44.003, 50.549, 0.000)
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False)
ERROR: Spawn failed because of collision at spawn position
Spawned 44 / 45 requested vehicles.
Spawned traffic vehicles: 44


In [ ]:
stop

NameError: name 'stop' is not defined

## 1) Create or load a Trial
A Trial owns its folder under `PATH_TO_TRIALS` and references a `route.json` that contains waypoint IDs.

In [ ]:
trial_name = "demo_trial_001"
trial = Trial(
    name=trial_name,
    map_name="Town03",
    weather="ClearNoon",
    description="Demo trial with predefined route.",
    n_vehicles=0,
    n_pedestrians=0,
 )
trial.save()
print(f"Trial saved at: {trial.folder_path}")
print(f"Route JSON: {trial.route_path}")

Trial saved at: d:\Projectes\TFG\MIREIA\trials\demo_trial_001
Route JSON: d:\Projectes\TFG\MIREIA\trials\demo_trial_001\route.json


## 2) Create a route (run once)
Use the interactive picker to select waypoints and save `route.json`. Skip this if the route already exists.

wm = WorldManager(verbose=True)
wm.load_scenario(trial)

route = create_route_from_waypoints(wm.bridge.get_waypoints())
save_route(route, trial.route_path)
plot_route(wm.bridge.get_waypoints(), [wp.id for wp in route.waypoints])

wm.destroy()
print(f"Route saved to: {trial.route_path}")

In [ ]:
run_id, run_path = trial.create_run_folder()
run_path = Path(run_path)
dataset_jsonl = run_path / "dataset.jsonl"
images_dir = run_path / "images"
images_dir.mkdir(parents=True, exist_ok=True)

static_meta = {
    "trial_name": trial.name,
    "run_id": run_id,
}
print(f"Run folder: {run_path}")
print(f"JSONL path: {dataset_jsonl}")

Run folder: d:\Projectes\TFG\MIREIA\trials\demo_trial_001\runs\20260323_221836
JSONL path: d:\Projectes\TFG\MIREIA\trials\demo_trial_001\runs\20260323_221836\dataset.jsonl


## 4) Run a trial
The current flow uses autopilot only. A route-following controller can be added later.

In [ ]:
use_controller = False  # False = autopilot
compute_model_risk = False
num_frames = 200
frame_stride = 5

wm = WorldManager(verbose=True)
wm.load_scenario(trial)
wm.enable_recording(
    append=False,
    include_topdown=False,
    include_static_risk_image=False,
    jsonl_path=str(dataset_jsonl),
    static_meta=static_meta,
 )
wm.setup_sensors(save_dir=str(images_dir), enable_map_camera=False)
delta_seconds = wm.world.get_settings().fixed_delta_seconds or 0.05

start_time = time.time()
for frame_id in range(num_frames):
    if frame_id % frame_stride != 0:
        wm.tick()
        continue

    rgb_path = images_dir / f"rgb_{frame_id:06d}.png"
    rel_rgb_path = str(rgb_path.relative_to(run_path))

    def tick_and_log():
        ctrl = wm.ego_vehicle.get_control() if wm.ego_vehicle is not None else None
        extra_fields = {
            "control": {
                "throttle": ctrl.throttle if ctrl else 0.0,
                "steer": ctrl.steer if ctrl else 0.0,
                "brake": ctrl.brake if ctrl else 0.0,
            },
            "mode": "controller" if use_controller else "autopilot",
        }
        wm.tick(
            ground_truth_risk=None,
            rgb_image_path=rel_rgb_path,
            extra_fields=extra_fields,
        )

    wm.sensor_manager.save_ego_frame(save_path=str(rgb_path), tick_fn=tick_and_log)

end_time = time.time()
elapsed = end_time - start_time
trial.add_run_summary(run_id, {"elapsed_seconds": elapsed, "frames": num_frames, "delta_seconds": delta_seconds})
print(f"Trial run completed in {elapsed:.1f}s")

wm.destroy()

Connecting to CARLA at 127.0.0.1:2000...
Connected. Current map: 'Carla/Maps/Town10HD_Opt'.
Loading scenario 'demo_trial_001' (map=Town03, seed=42)...
Switching map from 'Carla/Maps/Town10HD_Opt' to 'Town03'...
Weather set to 'ClearNoon'.
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=True)
SimulationBridge initialized: SimulationBridge(Ego: None, Dynamic Obstacles: 0, Static Obstacles: 7092, EnvState: (Visibility: 282.0m, Friction: 0.80))
Computing static risk map with bounds: (49.96485900878906, 0.818756103515625, 456.9049987792969) and resolution: 2.0m...
Scenario 'demo_trial_001' is ready.
Recording enabled → d:\Projectes\TFG\MIREIA\trials\demo_trial_001\runs\20260323_221836\dataset.jsonl


AttributeError: 'NoneType' object has no attribute 'x'

## 5) Append model risk (optional)
Run post-inference to add `model_risk` per frame. Skip for autopilot-only runs.

In [ ]:
if compute_model_risk:
    import json
    import torch
    from PIL import Image

    from MIREIA.data_collection.dataset_utils import (
        build_default_transform,
        load_jsonl_records,
        resolve_image_path,
    )
    from MIREIA.perception.e2e_model import E2EModelConfig, E2ERiskPredictor

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    transform = build_default_transform()

    def load_checkpoint(model: torch.nn.Module, checkpoint_path: str) -> None:
        state = torch.load(checkpoint_path, map_location=device)
        if isinstance(state, dict):
            if "model_state_dict" in state:
                state = state["model_state_dict"]
            elif "model_state" in state:
                state = state["model_state"]
            elif "state_dict" in state:
                state = state["state_dict"]
        model.load_state_dict(state)
        model.eval()

    def load_image(path: str) -> torch.Tensor:
        with Image.open(path) as img:
            img = img.convert("RGB")
            return transform(img)

    def predict_from_paths(model: torch.nn.Module, image_paths: list[str]) -> float:
        seq_tensor = torch.stack([load_image(path) for path in image_paths], dim=0)
        if seq_tensor.ndim == 4:
            seq_tensor = seq_tensor.unsqueeze(0)
        seq_tensor = seq_tensor.to(device)
        with torch.no_grad():
            pred = model(seq_tensor)
        return float(pred.squeeze().item())

    def append_model_risk_to_jsonl(
        jsonl_path: str,
        model: torch.nn.Module,
        seq_len: int,
        output_key: str = "model_risk",
        image_root: str | None = None,
        input_key: str = "rgb_image_path",
        normalize_paths: bool = True,
    ) -> None:
        if seq_len <= 0:
            raise ValueError("seq_len must be > 0")
        records = load_jsonl_records(jsonl_path)
        if not records:
            raise RuntimeError(f"No records found in {jsonl_path}")
        base_dir = image_root or os.path.dirname(jsonl_path)
        for idx, record in enumerate(records):
            if idx + 1 < seq_len:
                record[output_key] = None
                continue
            window = records[idx - seq_len + 1 : idx + 1]
            image_paths: list[str] = []
            for rec in window:
                rel_path = rec.get(input_key, "")
                if not rel_path:
                    raise ValueError(f"Missing {input_key} in {jsonl_path}")
                image_paths.append(resolve_image_path(base_dir, rel_path, normalize_paths))
            record[output_key] = predict_from_paths(model, image_paths)
        with open(jsonl_path, "w", encoding="utf-8") as handle:
            for record in records:
                handle.write(json.dumps(record) + "\n")

    checkpoint_path = os.path.join(Config.PATH_TO_MODELS, "single_checkpoint.pt")
    model = E2ERiskPredictor(E2EModelConfig()).to(device)
    load_checkpoint(model, checkpoint_path)
    append_model_risk_to_jsonl(
        jsonl_path=str(dataset_jsonl),
        model=model,
        seq_len=5,
        image_root=str(run_path),
    )
    print("Model risk appended to JSONL")
else:
    print("Skipping model risk")

## 6) Compose the video with risk chart
This renders a line chart tile with ground-truth risk and prediction if present.

In [ ]:
from MIREIA.analysis.visualizer import DatasetVideoComposer

composer = DatasetVideoComposer(
    str(dataset_jsonl),
    fps=10,
    risk_tile_mode="curve",
    delta_seconds=delta_seconds,
 )
video_path = composer.build_video(output_name="trial_video.mp4")
print(f"Saved video: {video_path}")

## 7) Compare two runs (overlay + AUC)
Set two JSONL paths from different runs to generate an overlay plot.

In [ ]:
# Example: compare the latest run with a previous run.
# Update these paths to match your runs.
run_a_jsonl = str(dataset_jsonl)
run_b_jsonl = ""  # e.g., r"D:/Projectes/TFG/MIREIA/trials/demo_trial_001/runs/20260322_214235/dataset.jsonl"

if run_b_jsonl:
    aucs = plot_risk_curves(
        [run_a_jsonl, run_b_jsonl],
        labels=["run_a", "run_b"],
        output_path=str(run_path / "risk_compare.png"),
        key="ground_truth_risk",
        delta_seconds=delta_seconds,
    )
    print("Saved overlay plot to", run_path / "risk_compare.png")
    print("AUCs:", aucs)
else:
    print("Set run_b_jsonl to compare runs")